# The World's Biggest Languages

Who are the top speakers on Earth, and what do macrolanguages mean for the count?
This notebook ranks global speaker totals from `low`, explores ISO scope codes, and
demonstrates the `LanguageCollection` lookup and filter API.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [ ]:
%pip install -q pandas plotly matplotlib


In [ ]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [ ]:
db = low.LanguagesOfTheWorld()
print(db)
print(f"Total languages: {len(db.languages)}")
print(f"First ten: {[l.label for l in db.languages[:10]]}")

## 2 · Polymorphic lookup

`db.languages.get()` accepts ISO 639-1 (2-char), ISO 639-3 (3-char), or a
case-insensitive label string.

In [ ]:
for query in ("rw", "kin", "Kinyarwanda"):
    lang = db.languages.get(query)
    print(f"get({query!r}) → {lang.label} ({lang.part3}, ISO 639-1: {lang.part1})")

## 3 · Filter by minimum speakers

`filter()` returns languages whose global `speaker_count` meets the threshold.

In [ ]:
popular = db.languages.filter(min_speakers=50_000_000)
print(f"Languages with ≥ 50 M speakers: {len(popular)}")

rows = [
    {
        "label": lang.label,
        "part3": lang.part3,
        "part1": lang.part1,
        "scope": lang.scope,
        "speaker_count": lang.speaker_count,
    }
    for lang in popular
]

df = pd.DataFrame(rows).sort_values("speaker_count", ascending=False)
df.head(15)

## 4 · Bar chart — top 30 languages (log scale)

In [ ]:
top30 = df.head(30)

fig = px.bar(
    top30,
    x="speaker_count",
    y="label",
    orientation="h",
    color="scope",
    text="speaker_count",
    log_x=True,
    labels={"speaker_count": "Speakers (log scale)", "label": "", "scope": "ISO scope"},
    title="Top 30 Languages by Global Speaker Count",
    color_discrete_sequence=px.colors.qualitative.Set2,
)

fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=700, margin=dict(l=10, r=10, t=40, b=10))
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.show()

## 5 · Macrolanguages vs individual codes

ISO 639-3 scope `"M"` marks a macrolanguage that groups related individual codes.
Compare Chinese (`zho`) with its most-spoken members.

In [ ]:
chinese = db.languages.get("zho")
print(f"{chinese.label}: scope={chinese.scope}, speakers={chinese.speaker_count:,}")

mandarin = db.languages.get("cmn")
cantonese = db.languages.get("yue")
for lang in (mandarin, cantonese):
    if lang:
        print(f"  {lang.label} ({lang.part3}): scope={lang.scope}, speakers={lang.speaker_count:,}")

## 6 · Scope distribution

In [ ]:
scope_labels = {"I": "Individual", "M": "Macrolanguage", "S": "Special"}

scope_df = pd.DataFrame([
    {"scope": scope_labels.get(lang.scope, lang.scope), "count": 1}
    for lang in db.languages
]).groupby("scope", as_index=False).sum()

fig2 = px.pie(
    scope_df,
    names="scope",
    values="count",
    title="ISO 639-3 Scope Distribution",
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig2.update_traces(textposition="inside", textinfo="percent+label")
fig2.show()

## 7 · Label substring filter

In [ ]:
portuguese_variants = db.languages.filter(label_contains="Portug")
print(f"Languages with 'Portug' in the label: {len(portuguese_variants)}")
for lang in portuguese_variants[:8]:
    sc = f"{lang.speaker_count:,}" if lang.speaker_count else "—"
    print(f"  {lang.label} ({lang.part3}): {sc} speakers")

## 8 · Summary

In [ ]:
with_speakers = sum(1 for l in db.languages if l.speaker_count)
print(f"Languages with a global speaker count: {with_speakers:,} / {len(db.languages):,}")
print(f"Macrolanguages: {sum(1 for l in db.languages if l.scope == 'M')}")
print(f"Individual languages: {sum(1 for l in db.languages if l.scope == 'I')}")